In [ ]:
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq

sys.path.insert(0, str(Path.cwd().parent.parent))   # racine du dépôt (dossier contenant Sources/)

from Sources import data as src_data                     # catalogue + charger()
from Sources import config
from Sources import signaux
from Sources import indicateurs
from Sources import features

catalogue = src_data.catalogue          # DataFrame indexé par nom lisible
charger   = src_data.charger            # chargement (cache local) d'un fichier

from IPython.display import Markdown

tab = catalogue.groupby(["source", "defaut"]).size().unstack(fill_value=0)
tab = tab[["Sain", "IR", "B", "OR"]].reindex(["Normal", "12kDE", "48kDE", "12kFE"])
tab = tab.reset_index().rename(columns={"source": "Source", "IR": "Bague int. (IR)",
                                        "B": "Bille (B)", "OR": "Bague ext. (OR)"})
# Markdown(tab.to_markdown(index=False))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
#| label: selection
#| code-summary: "Sélection des 40 fichiers"
#| tbl-cap: "Nombre de fichiers sélectionnés par localisation et diamètre du défaut."
# Chaque critère accepte une valeur ou un tuple ; None = pas de filtre.
# Ici : 12 kHz côté entraînement, défauts IR/B/OR de 0,007" à 0,021",
# bague extérieure en position 6 h, charges 0 à 3 HP, + les 4 sains.
selection = src_data.selectionner(sources=("12kDE","12kFE",),
                                  defauts=("IR", "B", "OR"),
                                  diametres=(7, 14, 21),
                                  positions_or=("6",),
                                  charges=(0, 1, 2, 3),
                                  avec_sains=True)
# selection.groupby(["defaut", "diametre_mils"]).size().unstack(fill_value=0)

In [17]:
selection

,numero,source,fs,defaut,position_OR,diametre_mils,charge_hp,rpm_nominal
nom,,,,,,,,
Normal_L0,97,Normal,12000,Sain,NaN,0,0,1797
Normal_L1,98,Normal,12000,Sain,NaN,0,1,1772
Normal_L2,99,Normal,12000,Sain,NaN,0,2,1750
Normal_L3,100,Normal,12000,Sain,NaN,0,3,1730
12kDE_IR007_L0,105,12kDE,12000,IR,NaN,7,0,1797
...,...,...,...,...,...,...,...,...
12kFE_B021_L0,290,12kFE,12000,B,NaN,21,0,1797
12kFE_B021_L1,291,12kFE,12000,B,NaN,21,1,1772
12kFE_B021_L2,292,12kFE,12000,B,NaN,21,2,1750


In [ ]:
table = features.construire_table(selection, charger)
features.sauvegarder_table(table)                # écrit data/features/table_indicateurs.csv


INDICATEURS = indicateurs.COLONNES_TEMPORELS   # 4 indicateurs temporels (partie exploratoire)
ORDRE = ["Sain", "IR007", "IR014", "IR021", "B007", "B014", "B021",
         "OR007", "OR014", "OR021"]

In [16]:
table.groupby(["classe", "charge_hp"]).size().unstack()


charge_hp,0,1,2,3
classe,,,,
B007,101,100,100,100
B014,100,100,100,100
B021,100,100,100,100
IR007,100,100,100,101
IR014,100,100,100,100
IR021,100,100,100,100
OR007,100,101,100,101
OR014,100,50,50,50
OR021,101,50,50,50


In [13]:
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."

X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_strat(
    table, indicateurs.COLONNES_INDICATEURS)

pd.DataFrame({
    "split": ["train", "test"],
    "fichiers": [g_train.nunique(), g_test.nunique()],
    "fenêtres": [len(X_train), len(X_test)],
    "classes présentes": [", ".join(sorted(y_train.unique())),
                          ", ".join(sorted(y_test.unique()))],
})

,split,fichiers,fenêtres,classes présentes
0,train,30,1858,"B007, B014, B021, IR007, IR014, IR021, OR007, ..."
1,test,10,653,"B007, B014, B021, IR007, IR014, IR021, OR007, ..."
